# YUNESA Academic Knowledge Graph Construction

Notebook ini adalah entrypoint canonical untuk Academic Knowledge Graph construction. Source utama ada di `src/yunesa_academic_kg.py`; modul lama seperti `graph_builder.py`, `embedding.py`, `graphrag.py`, `nlp_parser.py`, dan `data_loader.py` hanya dipertahankan untuk kompatibilitas notebook historis.

Notebook ini membangun Academic Knowledge Graph dari data `papers`, `lecturers`, dan `paper_lecturers` di Supabase, lalu memperkaya node `Concept` memakai IEEE taxonomy/thesaurus sebagai controlled vocabulary.

Workflow notebook:

1. Load sample data paper dan dosen dari Supabase.
2. Load IEEE SKOS taxonomy/thesaurus sebagai controlled vocabulary.
3. Bangun backbone graph: `Lecturer`, `Publication`, `Venue`, `Year`, `Institution`, dan `Keyword`.
4. Ekstrak `Concept` sesuai ontology Bab 3: `Problem`, `ResearchTopic`, `Task`, `Domain`, `Method`, `Model`, `Dataset`, `Metric`, `Result`, dan `Innovation`.
5. Hubungkan publikasi ke concept memakai relasi ontology English: `HAS_TOPIC`, `USES_METHOD`, `USES_MODEL`, `USES_DATASET`, `EVALUATED_WITH`, `HAS_RESULT`, dan `BELONGS_TO_DOMAIN`.
6. Turunkan relasi author dan co-authorship: `HAS_AUTHOR`, `PUBLISHES`, dan `COLLABORATES_WITH`.
7. Tambahkan relasi SKOS satu-hop dari IEEE vocabulary untuk memperkaya konteks concept tanpa memaksa semua concept menjadi IEEE term.
8. Export hasil ke CSV, JSON node-link, GraphML, dan opsional tulis ke Neo4j AuraDB serta Zilliz/Milvus.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys

def is_colab_runtime() -> bool:
    return (
        'google.colab' in sys.modules
        or bool(os.getenv('COLAB_RELEASE_TAG'))
        or importlib.util.find_spec('google.colab') is not None
    )

IN_COLAB = is_colab_runtime()
BOOTSTRAP_VERSION = "2026-06-03-kg-gliner-glirel-minimal-copy"
print("Notebook bootstrap version:", BOOTSTRAP_VERSION)
DEFAULT_DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Tugas_Akhir')
PROJECT_DIR = Path(os.getenv('YUNESA_PROJECT_DIR', '')).expanduser() if os.getenv('YUNESA_PROJECT_DIR') else None
USE_GIT_CLONE = os.getenv('YUNESA_USE_GIT_CLONE', '0') == '1'
REPO_URL = os.getenv('YUNESA_REPO_URL', 'https://github.com/rizkyyanuark/Tugas_Akhir.git')
REPO_BRANCH = os.getenv('YUNESA_REPO_BRANCH', 'master')
REPO_DIR = Path('/content/Tugas_Akhir')
RUNTIME_PROJECT_DIR = Path('/content/Tugas_Akhir_runtime')
USE_RUNTIME_COPY = os.getenv('YUNESA_USE_RUNTIME_COPY', '1') == '1'

def has_kg_sources(project_dir: Path) -> bool:
    return (
        project_dir / 'notebooks' / 'build-graph' / 'src' / 'yunesa_academic_kg.py'
    ).exists() and (
        project_dir / 'notebooks' / 'build-graph' / 'ieee-thesaurus.ttl'
    ).exists()

def copy_project_to_runtime(project_dir: Path) -> Path:
    if not IN_COLAB or not USE_RUNTIME_COPY:
        return project_dir
    if str(project_dir).startswith(str(RUNTIME_PROJECT_DIR)):
        return project_dir

    # Copy only the assets required by this notebook. Copying the full repo from
    # Google Drive is fragile in Colab and may fail on unrelated backend files.
    if RUNTIME_PROJECT_DIR.exists():
        shutil.rmtree(RUNTIME_PROJECT_DIR)
    print(f'Copying KG notebook assets to Colab runtime: {RUNTIME_PROJECT_DIR}')

    required_dirs = [
        Path('notebooks') / 'build-graph',
    ]
    optional_dirs = [
        Path('data') / 'manual_tests',
    ]
    optional_files = [
        Path('notebooks') / 'pyproject.toml',
        Path('notebooks') / 'uv.lock',
        Path('.env'),
    ]
    ignore = shutil.ignore_patterns('__pycache__', '.ipynb_checkpoints', 'outputs', '*.pyc', '*.pyo')

    for rel in required_dirs:
        src_path = project_dir / rel
        dst_path = RUNTIME_PROJECT_DIR / rel
        if not src_path.exists():
            raise RuntimeError(f'Required KG asset missing on Drive: {src_path}')
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src_path, dst_path, ignore=ignore, dirs_exist_ok=True)

    for rel in optional_dirs:
        src_path = project_dir / rel
        dst_path = RUNTIME_PROJECT_DIR / rel
        if src_path.exists():
            dst_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src_path, dst_path, ignore=ignore, dirs_exist_ok=True)

    for rel in optional_files:
        src_path = project_dir / rel
        dst_path = RUNTIME_PROJECT_DIR / rel
        if src_path.exists():
            dst_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_path, dst_path)

    return RUNTIME_PROJECT_DIR

if IN_COLAB:
    if PROJECT_DIR is None or not has_kg_sources(PROJECT_DIR):
        try:
            from google.colab import drive
            if not Path('/content/drive').exists() or not any(Path('/content/drive').iterdir()):
                drive.mount('/content/drive')
        except Exception as exc:
            print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')

    if PROJECT_DIR and has_kg_sources(PROJECT_DIR):
        os.chdir(PROJECT_DIR / 'notebooks' / 'build-graph')
    elif has_kg_sources(DEFAULT_DRIVE_PROJECT_DIR):
        PROJECT_DIR = DEFAULT_DRIVE_PROJECT_DIR
        os.chdir(PROJECT_DIR / 'notebooks' / 'build-graph')
    elif USE_GIT_CLONE:
        if not REPO_DIR.exists():
            subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
        else:
            subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH, '--depth', '1'])
            subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'])
        PROJECT_DIR = REPO_DIR
        os.chdir(PROJECT_DIR / 'notebooks' / 'build-graph')
    else:
        raise RuntimeError(
            'KG source files are not available in Colab runtime. '
            'Upload/sync the repo folder to Google Drive as MyDrive/Tugas_Akhir, '
            'or set YUNESA_PROJECT_DIR, or set YUNESA_USE_GIT_CLONE=1.'
        )
    if USE_RUNTIME_COPY and PROJECT_DIR != REPO_DIR:
        PROJECT_DIR = copy_project_to_runtime(PROJECT_DIR)
        os.chdir(PROJECT_DIR / 'notebooks' / 'build-graph')
    pip_packages = ['pandas', 'networkx', 'rdflib==6.3.1', 'supabase>=2.25.1', 'python-dotenv', 'opik>=1.0.0', 'pyvis', 'neo4j>=6.1.0', 'pymilvus>=2.6.5', 'requests']
    required_modules = ['pandas', 'networkx', 'rdflib', 'supabase', 'dotenv', 'opik', 'pyvis', 'neo4j', 'pymilvus', 'requests']
else:
    pip_packages = []
    required_modules = ['pandas', 'networkx', 'rdflib', 'supabase', 'dotenv', 'opik', 'neo4j', 'pymilvus', 'requests']

def missing_modules(module_names: list[str]) -> list[str]:
    return [pkg for pkg in module_names if importlib.util.find_spec(pkg) is None]

missing = missing_modules(required_modules)
if missing and IN_COLAB:
    print('Installing notebook dependencies: ' + ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pip_packages])
    missing = missing_modules(required_modules)

if missing:
    raise RuntimeError(
        'Notebook dependencies are missing: ' + ', '.join(missing) +
        '. Local VS Code: run `uv sync --project notebooks`, then select the `notebooks/.venv` kernel. '
        'Colab/VS Code Colab: reconnect to a Colab runtime and rerun this cell.'
    )

BUILD_GRAPH_DIR = Path.cwd()
if not (BUILD_GRAPH_DIR / 'src').exists():
    BUILD_GRAPH_DIR = Path('notebooks/build-graph').resolve()

sys.path.insert(0, str(BUILD_GRAPH_DIR / 'src'))

from yunesa_academic_kg import (
    KGConfig,
    IeeeSemanticIndex,
    AcademicKGBuilder,
    AcademicExtractionConfig,
    academicrag_storage_plan,
    build_academicrag_document_records,
    build_milvus_index_records,
    export_graph_artifacts,
    extract_academic_elements_with_gliner_glirel,
    fetch_supabase_sample,
    graph_to_frames,
    load_local_csv_sample,
    load_project_env,
    milvus_credential_status,
    neo4j_credential_status,
    summarize_academicrag_document_records,
    summarize_extracted_elements,
    summarize_milvus_records,
    supabase_credential_status,
    write_dual_index_to_storage,
    write_graph_to_neo4j,
    write_vector_index_to_milvus,
)

config = KGConfig.default(sample_size=50)
load_project_env(config.project_root)
print('Supabase credential status:', supabase_credential_status())
print('Neo4j credential status:', neo4j_credential_status())
print('Milvus credential status:', milvus_credential_status())
config


## 1. Load Data

Default loader mengambil data dari Supabase. Jika credential belum tersedia atau koneksi gagal, cell ini otomatis fallback ke CSV lokal di `notebooks/scraping/file_tabulars`.

In [ ]:
try:
    papers_df, lecturers_df, links_df = fetch_supabase_sample(sample_size=config.sample_size)
    data_source = 'supabase'
except Exception as exc:
    print(f'Supabase load failed, using local CSV fallback: {type(exc).__name__}: {exc}')
    papers_df, lecturers_df, links_df = load_local_csv_sample(
        config.project_root / 'notebooks' / 'scraping' / 'file_tabulars',
        sample_size=config.sample_size,
    )
    data_source = 'local_csv'

print('Data source:', data_source)
print('Papers:', len(papers_df))
print('Lecturers:', len(lecturers_df))
print('Paper-lecturer links:', len(links_df))
papers_df.head(3)


## 2. Load IEEE Semantic Index

Index ini membaca `ieee-thesaurus.ttl` dan `ieee-taxonomy.ttl`. IEEE labels dipakai untuk grounding concept, bukan sebagai satu-satunya sumber concept. Keyword mentah dan regex teknis tetap dipakai agar istilah spesifik seperti model, dataset, dan metric tidak hilang.

In [ ]:
ieee_index = IeeeSemanticIndex.from_files(
    config.thesaurus_path,
    config.taxonomy_path,
    max_terms=config.max_ieee_terms,
)
ieee_index.summary()


## 3. Optional GLiNER/GLiREL Element Extraction

AcademicRAG memakai LLM untuk entity, relationship, dan keyword extraction. Pada tugas akhir ini, jalur teks tidak terstruktur dapat diganti dengan GLiNER untuk zero-shot NER dan GLiREL untuk zero-shot relation extraction.

Default-nya tidak aktif agar notebook tidak otomatis mengunduh model besar. Aktifkan dengan:

```python
os.environ['YUNESA_USE_GLINER'] = '1'
os.environ['YUNESA_USE_GLIREL'] = '1'
```


In [ ]:
extraction_config = AcademicExtractionConfig.from_env()
print('Extraction config:', extraction_config.status())

optional_packages = []
if extraction_config.use_gliner:
    optional_packages.extend(['gliner>=0.2.26', 'torch'])
if extraction_config.use_glirel:
    optional_packages.extend(['glirel>=1.2.1', 'spacy>=3.8.14'])

if optional_packages and IN_COLAB:
    print('Installing optional extraction dependencies:', optional_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *optional_packages])
elif optional_packages and not IN_COLAB:
    missing_optional = [pkg.split('>=')[0] for pkg in optional_packages if importlib.util.find_spec(pkg.split('>=')[0].replace('-', '_')) is None]
    if missing_optional:
        print('Optional extractor dependencies are missing locally:', missing_optional)
        print('Install them in notebooks env or disable with YUNESA_USE_GLINER=0 and YUNESA_USE_GLIREL=0.')

try:
    extracted_elements = extract_academic_elements_with_gliner_glirel(
        papers_df,
        extraction_config,
    )
except Exception as exc:
    extracted_elements = {}
    print(f'Optional GLiNER/GLiREL extraction skipped: {type(exc).__name__}: {exc}')
    print('Graph construction will continue with deterministic IEEE/keyword/regex extraction.')

print('Extracted element summary:', summarize_extracted_elements(extracted_elements))


## 4. Build Knowledge Graph

Graph dibangun sebagai `networkx.MultiDiGraph`, sehingga satu pasang node bisa memiliki beberapa edge dengan relasi berbeda. Setiap edge concept menyimpan `source`, `match_type`, `matched_text`, `score`, dan `provenance`.

In [ ]:
builder = AcademicKGBuilder(ieee_index, extracted_elements=extracted_elements)
G = builder.build(
    papers_df=papers_df,
    lecturers_df=lecturers_df,
    links_df=links_df,
    max_concepts_per_paper=config.max_concepts_per_paper,
)

validation = builder.validate()
validation


## 5. Inspect Nodes and Edges

In [ ]:
nodes_df, edges_df = graph_to_frames(G)
display(nodes_df.groupby('node_type').size().sort_values(ascending=False).to_frame('count'))
display(edges_df.groupby('relation').size().sort_values(ascending=False).to_frame('count'))


In [ ]:
concept_edges = edges_df[edges_df['relation'].isin([
    'HAS_TOPIC',
    'USES_METHOD',
    'USES_MODEL',
    'USES_DATASET',
    'EVALUATED_WITH',
    'HAS_RESULT',
    'BELONGS_TO_DOMAIN',
])]
concept_edges[['source', 'target', 'relation', 'edge_source', 'match_type', 'matched_text', 'score']].head(20)


## 6. Export Artifacts

Output disimpan di `notebooks/build-graph/outputs/academic_kg/`:

- `academic_kg_nodes.csv`
- `academic_kg_edges.csv`
- `academic_kg_node_link.json`
- `academic_kg.graphml`
- `academic_kg_summary.json`

In [ ]:
artifacts = export_graph_artifacts(G, config.output_dir)
for name, path in artifacts.items():
    print(f'{name}: {path}')


## 7. Optional Neo4j AuraDB Load

Cell ini menulis property graph ke Neo4j/AuraDB. Secara default tidak otomatis menulis agar aman saat eksplorasi.

Set environment berikut sebelum menjalankan cell jika ingin menulis ke AuraDB:

- `YUNESA_WRITE_NEO4J=1`
- `YUNESA_NEO4J_CLEAR_EXISTING=1` jika ingin rebuild graph bersih untuk `graph_name` yang sama.


In [ ]:
print('Neo4j credential status:', neo4j_credential_status())
WRITE_NEO4J = os.getenv('YUNESA_WRITE_NEO4J', '0') == '1'
NEO4J_GRAPH_NAME = os.getenv('YUNESA_GRAPH_NAME', os.getenv('YUNESA_NEO4J_GRAPH_NAME', 'yunesa_academic_kg_sample'))
NEO4J_CLEAR_EXISTING = os.getenv('YUNESA_NEO4J_CLEAR_EXISTING', '0') == '1'

if WRITE_NEO4J:
    neo4j_report = write_graph_to_neo4j(
        G,
        graph_name=NEO4J_GRAPH_NAME,
        clear_existing=NEO4J_CLEAR_EXISTING,
    )
    print('Neo4j write report:', neo4j_report)
else:
    print('Neo4j write skipped. Set YUNESA_WRITE_NEO4J=1 to enable.')


## 8. Optional Milvus/Zilliz Vector Index Load

Cell ini membuat dual indexing ala AcademicRAG: graph tetap di Neo4j, sedangkan retrieval semantic masuk ke Milvus/Zilliz.

Koleksi yang disiapkan:

- `PaperChunk`: title, TLDR, abstract, keyword, concept, author.
- `EntityEmbedding`: node/entity description.
- `RelationshipEmbedding`: deskripsi triple/edge graph.
- `ContentKeyword`: keyword dan concept terkontrol per paper.

Secara default cell hanya preview jumlah row. Set `YUNESA_WRITE_MILVUS=1` untuk insert ke Zilliz. Gunakan `YUNESA_GRAPH_NAME` agar Neo4j dan Zilliz memakai namespace graph yang sama. Default embedding memakai SiliconFlow `Qwen/Qwen3-Embedding-0.6B` dengan dimensi 1024; UI agent harus memakai model/dimensi yang sama. `YUNESA_MILVUS_CLEAR_EXISTING=1` membersihkan row untuk `graphName` tersebut, bukan drop semua collection. Jika mengganti model embedding, rebuild bersih atau gunakan graph name baru.


In [ ]:
print(json.dumps(academicrag_storage_plan(), indent=2))
print('Milvus credential status:', milvus_credential_status())

GRAPH_NAME = os.getenv('YUNESA_GRAPH_NAME', os.getenv('YUNESA_NEO4J_GRAPH_NAME', 'yunesa_academic_kg_sample'))
print('Graph namespace:', GRAPH_NAME)

document_records = build_academicrag_document_records(G)
print('Prepared document records:', summarize_academicrag_document_records(document_records))
vector_records = build_milvus_index_records(G, graph_name=GRAPH_NAME)
print('Prepared vector rows:', summarize_milvus_records(vector_records))

WRITE_MILVUS = os.getenv('YUNESA_WRITE_MILVUS', '0') == '1'
MILVUS_CLEAR_EXISTING = os.getenv('YUNESA_MILVUS_CLEAR_EXISTING', '0') == '1'

if WRITE_MILVUS:
    milvus_report = write_vector_index_to_milvus(
        G,
        clear_existing=MILVUS_CLEAR_EXISTING,
        graph_name=GRAPH_NAME,
    )
    print(json.dumps(milvus_report, indent=2, default=str))
else:
    print('Milvus write skipped. Set YUNESA_WRITE_MILVUS=1 to enable.')
    print('For a clean rebuild, also set YUNESA_MILVUS_CLEAR_EXISTING=1.')


## 9. Optional Visualization

Cell ini membuat HTML interaktif memakai PyVis. Jalankan hanya untuk sample kecil agar browser tetap ringan.

In [ ]:
# Optional: generate an interactive HTML graph for small samples.
from pyvis.network import Network

preview_path = config.output_dir / 'academic_kg_preview.html'
net = Network(height='760px', width='100%', bgcolor='#ffffff', directed=True, notebook=True)

color_by_type = {
    'Lecturer': '#2f80ed',
    'Publication': '#27ae60',
    'Venue': '#8e44ad',
    'Year': '#7f8c8d',
    'Institution': '#16a085',
    'Keyword': '#f39c12',
    'Concept': '#c0392b',
}

for node_id, data in G.nodes(data=True):
    node_type = data.get('node_type', 'Unknown')
    net.add_node(
        node_id,
        label=str(data.get('label', node_id))[:60],
        title=f"{node_type}: {data.get('label', node_id)}",
        color=color_by_type.get(node_type, '#95a5a6'),
    )

for source, target, data in G.edges(data=True):
    net.add_edge(source, target, label=data.get('relation', ''), title=data.get('relation', ''))

net.show(str(preview_path))
preview_path
